# Assignment — Bloc 5a, Sessió 8
### Mini-repte: Oscillator, Envelope, SamplePlayer i la teva Veu

**Com funciona:** Aquest notebook és la teva còpia personal, generada des de la teva tasca del Classroom. Els canvis es desen automàticament al teu Drive mentre treballes. Quan acabis, torna a la tasca del Classroom i prem **Lliura**. No cal fer «Desa una còpia»: ja estàs treballant sobre la teva.

**Per què a Colab i no a Thonny?** Avui generem el so com a arrays sencers: no hi ha cap callback ni cap rellotge en temps real. Colab ens deixa anar cel·la a cel·la, mirar què hi ha dins de cada objecte, dibuixar i escoltar al moment. A la Sessió 9 posarem aquestes mateixes classes dins d'un callback, a Thonny.

**Què faràs:**
0. Escalfament: dos objectes, dues memòries
1. `Oscillator`: les formes d'ona que falten i la memòria de fase (`_phase`)
2. `Envelope`: ja feta — l'escoltes i la proves
3. `SamplePlayer`: **la primera classe que escrius sencera**
4. `Veu`: un instrument fet de peces (un `Oscillator` + un `Envelope` a dins)
5. Munta-ho i escolta-ho: la teva seqüència

Tingues a mà el full de l'exercici en paper: les caixes que hi has dibuixat t'ajudaran a la Part 3.

No esborris les cel·les d'autotest.

In [ ]:
import numpy as np
from scipy import signal
import soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import Audio, display
import urllib.request

sample_rate = 44100

base_url = "https://raw.githubusercontent.com/jcomajuncosas/tp2/main/05_sintesi/sessio08_oscil_ladors_classes/recursos/audio/"
urllib.request.urlretrieve(base_url + "kick_sample.wav", "kick_sample.wav")
print("Fitxers carregats")

## Part 0 — Escalfament: dos objectes, dues memòries

Abans d'executar la cel·la, **escriu en un comentari què creus que s'imprimirà**. Després executa-la i comprova-ho.

In [ ]:
class Fader:
    def __init__(self, nivell=1.0):
        self.nivell = nivell            # cada instància guarda el SEU nivell

    def aplica(self, so):
        return so * self.nivell

f1 = Fader(1.0)
f2 = Fader(0.2)
f1.nivell = 0.5                         # canviem NOMÉS f1

# La meva predicció: ...

print("f1:", f1.nivell)
print("f2:", f2.nivell)

L'error que més veuràs avui: oblidar `self.` en guardar una dada. Llegeix aquesta classe, **descomenta l'última línia**, executa i llegeix el missatge d'error amb calma. Després torna-la a comentar.

In [ ]:
class FaderDespistat:
    def __init__(self, nivell=1.0):
        nivell = nivell                 # ⚠️ falta self. → la dada es perd en acabar __init__

    def aplica(self, so):
        return so * self.nivell

f3 = FaderDespistat(0.5)
# f3.aplica(np.ones(4))

## Part 1 — `Oscillator`, ara amb memòria

A la S05 la fase de l'oscil·lador vivia en una global (`fase_sinus`), i cada oscil·lador nou en necessitava una altra (`fase_ringmod`). Aquí la fase viu **dins** de l'objecte: `self._phase`. Cada `Oscillator()` que creïs portarà la seva.

El **guió baix** de `_phase` és un conveni de Python: vol dir *"estat intern, no el toquis des de fora"*. En canvi, `freq` o `waveform` sí que els pots canviar des de fora (`osc.freq = 880`).

- **TODO 1:** afegeix les formes d'ona `'square'` i `'sawtooth'`, per analogia amb `'sine'`.
- **TODO 2:** actualitza `self._phase` perquè la propera crida continuï on ha acabat aquesta.

In [ ]:
class Oscillator:
    def __init__(self, freq=440.0, waveform='sine', sample_rate=44100):
        self.freq = freq                # paràmetres: es poden canviar des de fora
        self.waveform = waveform
        self.sample_rate = sample_rate
        self._phase = 0.0               # estat intern: on som del cicle (0 … 2π)

    def generate(self, duration):
        n = int(round(self.sample_rate * duration))
        increment = 2 * np.pi * self.freq / self.sample_rate
        fases = self._phase + increment * np.arange(n)      # continuem on ho vam deixar

        # TODO 2: guarda a self._phase on ha de començar la PROPERA crida.
        #         És la mateixa fórmula de fase_sinus a la S05:
        #         la fase actual + increment * n, i després % (2 * np.pi)

        if self.waveform == 'sine':
            return np.sin(fases)
        # TODO 1: afegeix aquí les branques 'square' i 'sawtooth' (amb elif),
        #         per analogia amb 'sine'. Pistes: signal.square(fases), signal.sawtooth(fases)
        else:
            print("Forma d'ona desconeguda:", self.waveform, "→ retorno silenci")
            return np.zeros(n)

In [ ]:
# AUTOTEST 1
osc = Oscillator(freq=440, waveform='sine')
w = osc.generate(1.0)
assert w is not None, "generate() retorna None"
assert len(w) == 44100, f"S'esperaven 44100 mostres, n'hi ha {len(w)}"
assert np.max(np.abs(w)) > 0.9, "L'amplitud sembla incorrecta per a un sinus"

sq = Oscillator(220, 'square').generate(0.5)
assert len(sq) == 22050, "La quadrada no té la longitud esperada"
assert np.allclose(np.abs(sq), 1.0), "Una ona quadrada només pot valer +1 o -1"

sw = Oscillator(220, 'sawtooth').generate(0.5)
assert len(sw) == 22050, "La dent de serra no té la longitud esperada"
assert np.min(sw) < -0.9 and np.max(sw) > 0.9, "La dent de serra hauria d'anar de -1 a +1"
assert not np.allclose(np.abs(sw), 1.0), "Això sembla una quadrada, no una dent de serra"

# Memòria de fase: dos trossos seguits han d'encaixar sense salt
osc = Oscillator(freq=440)
tros1 = osc.generate(512 / 44100)
tros2 = osc.generate(512 / 44100)
pas_normal = 2 * np.pi * 440 / 44100      # el salt màxim entre dues mostres d'aquest sinus
assert abs(tros2[0] - tros1[-1]) < 2 * pas_normal, \
    "Hi ha un salt entre trossos: self._phase no s'actualitza (TODO 2)"

print("✅ Part 1 correcta!")

### 🔊 Prova-ho i escolta

In [ ]:
# Les tres formes d'ona, a la mateixa freqüència
for forma in ['sine', 'square', 'sawtooth']:
    print(forma)
    display(Audio(Oscillator(220, forma).generate(1.0), rate=sample_rate))

# Tros a tros, com ho demanarà un callback: 200 trossos de 512 mostres
osc = Oscillator(freq=440)
trossos = []
for i in range(200):
    trossos.append(osc.generate(512 / sample_rate))
print("200 trossos enganxats:")
display(Audio(np.concatenate(trossos), rate=sample_rate))

# 👉 Prova: comenta la línia del TODO 2, torna a executar la cel·la de la classe
#    i aquesta. Què sents ara? Després deixa-ho com estava.

## Part 2 — `Envelope` (ADSR): ja feta

L'envolvent dona **forma temporal** a la nota: com neix (Attack, Decay), com es manté (Sustain, que és un *nivell*) i com mor (Release). No hi ha res a completar: executa-la i juga amb els paràmetres de la cel·la següent.

Una aproximació a tenir present: aquí l'envolvent **sap per endavant** quant durarà la nota. En temps real no ho sabrem (el `note_off` arriba quan arriba): ho resoldrem a la Sessió 9.

In [ ]:
class Envelope:
    def __init__(self, attack=0.01, decay=0.1, sustain=0.7, release=0.2):
        self.attack = attack
        self.decay = decay
        self.sustain = sustain          # és un NIVELL (0-1), no un temps
        self.release = release

    def generate(self, note_duration, sample_rate=44100):
        """note_duration: temps que la nota està premuda (el release s'afegeix DESPRÉS)."""
        n_attack  = int(self.attack  * sample_rate)
        n_decay   = int(self.decay   * sample_rate)
        n_sustain = max(0, int(note_duration * sample_rate) - n_attack - n_decay)
        n_release = int(self.release * sample_rate)

        env_attack  = np.linspace(0, 1, n_attack)
        env_decay   = np.linspace(1, self.sustain, n_decay)
        env_sustain = np.full(n_sustain, self.sustain)
        env_release = np.linspace(self.sustain, 0, n_release)

        return np.concatenate([env_attack, env_decay, env_sustain, env_release])

In [ ]:
# 🔊 Prova-ho i escolta: canvia attack / decay / sustain / release i torna a executar
env = Envelope(attack=0.05, decay=0.15, sustain=0.5, release=0.4)
corba = env.generate(note_duration=0.6)

plt.figure(figsize=(10, 3))
plt.plot(np.arange(len(corba)) / sample_rate, corba)
plt.title("La teva envolvent ADSR")
plt.xlabel("temps (s)")
plt.ylabel("amplitud")
plt.show()

osc = Oscillator(freq=330, waveform='sawtooth')
ona = osc.generate(len(corba) / sample_rate)     # tantes mostres com la corba
print("Sense envolvent:")
display(Audio(ona, rate=sample_rate))
print("Amb envolvent:")
display(Audio(ona * corba, rate=sample_rate))

## Part 3 — `SamplePlayer`: la teva primera classe sencera

Ara l'escrius tu **des de zero**. Primer l'esquelet (com al paper), després omple els mètodes.

**Signatura i comportament:**

| | |
|---|---|
| Constructor | `SamplePlayer(filepath, gain=1.0, sample_rate=44100)` |
| Atributs | `self.sample` (l'array llegit del fitxer), `self.gain`, `self.sample_rate` |
| Mètode | `generate(duration)` → array d'**exactament** `duration` segons: el sample × `gain`. Si el sample és més llarg, retallat; si és més curt, amb silenci al final |

**Pistes:**
- Llegir un WAV: `data, sr = sf.read(filepath)` (Sessió 2)
- Nombre de mostres: `n = int(round(self.sample_rate * duration))`
- Retallar: `self.sample[:n]` (les `n` primeres mostres)
- Silenci: `np.zeros(quantes)`, i enganxar amb `np.concatenate([a, b])`

**Per què `generate` i no `play`?** Perquè fa la mateixa feina que `Oscillator.generate`: retornar `duration` segons de so. Mateixa feina → mateix nom. A la cel·la 🔊 veuràs per què és tan pràctic.

In [ ]:
# TODO 3: escriu aquí la classe SamplePlayer sencera

In [ ]:
# AUTOTEST 2
kick = SamplePlayer('kick_sample.wav')
assert len(kick.sample) == 13230, "self.sample hauria de contenir les 13230 mostres del fitxer"
assert kick.gain == 1.0, "El gain per defecte hauria de ser 1.0"

llarg = kick.generate(0.5)
assert llarg is not None, "generate() retorna None"
assert len(llarg) == 22050, f"0.5 s haurien de ser 22050 mostres, n'hi ha {len(llarg)}"
assert np.allclose(llarg[:13230], kick.sample), "Al principi hi ha d'haver el sample"
assert np.allclose(llarg[13230:], 0.0), "Després del sample hi ha d'haver silenci"

curt = kick.generate(0.1)
assert len(curt) == 4410, "Si el pas és més curt que el sample, cal retallar-lo"

fluix = SamplePlayer('kick_sample.wav', gain=0.3)
assert np.isclose(np.max(np.abs(fluix.generate(0.5))), 0.3 * np.max(np.abs(llarg))), \
    "El gain no s'aplica correctament (quan el sample és més curt que el pas)"
assert np.isclose(np.max(np.abs(fluix.generate(0.1))), 0.3 * np.max(np.abs(curt))), \
    "El gain no s'aplica correctament (quan es retalla el sample)"

print("✅ Part 3 correcta!")

In [ ]:
# 🔊 Prova-ho i escolta
kick = SamplePlayer('kick_sample.wav', gain=0.8)

# Un compàs de 4 negres a 120 bpm: cada pas dura 0.5 s
compas = []
for i in range(4):
    compas.append(kick.generate(0.5))
print("Quatre cops de bombo:")
display(Audio(np.concatenate(compas), rate=sample_rate))

# Mateix nom de mètode → el bucle no necessita saber qui és qui
fonts = [Oscillator(55, 'sawtooth'), kick]
mescla = np.zeros(int(0.5 * sample_rate))
for font in fonts:
    mescla = mescla + font.generate(0.5)
mescla = mescla / np.max(np.abs(mescla))
print("Un oscil·lador i un sample, mesclats pel mateix bucle:")
display(Audio(mescla, rate=sample_rate))

# 👉 Canvia kick.gain directament (kick.gain = 0.2) i torna a escoltar el compàs.

## Part 4 — `Veu`: un instrument fet de peces

A la demo, `tocar_nota()` era una **funció** que, a cada nota, creava un oscil·lador i una envolvent nous i els llençava. Un instrument de debò es construeix **un cop** (en tries el timbre) i després toca moltes notes.

Converteix-ho en una classe: una `Veu` **conté** un `Oscillator` i un `Envelope` (objectes dins d'objectes: *composició*), i a més respon a la **velocity**.

| | |
|---|---|
| Constructor | `Veu(waveform='sine', attack=0.02, decay=0.1, sustain=0.6, release=0.2)` → crea `self.osc` i `self.env` **un sol cop** |
| Mètode | `nota(note_number, note_duration, velocity=100)` → array |

**Els 4 passos de `nota`:**
1. Posa la freqüència de `self.osc` segons la nota. Pista: accedeix directament a l'atribut, `self.osc.freq = note_to_freq(...)`
2. Genera la corba amb `self.env`
3. Genera **exactament tantes mostres** d'oscil·lador com en té la corba (pista: `len(corba) / self.osc.sample_rate` segons)
4. Retorna ona × corba × (velocity / 127)

Si veus l'error `operands could not be broadcast together with shapes...`, l'ona i la corba no tenen la mateixa longitud: revisa el pas 3.

In [ ]:
def note_to_freq(note_number):
    """Número MIDI → freqüència (Hz). Ja la coneixes del Bloc 4."""
    return 440.0 * (2 ** ((note_number - 69) / 12))


class Veu:
    def __init__(self, waveform='sine', attack=0.02, decay=0.1, sustain=0.6, release=0.2):
        # TODO 4: crea self.osc (un Oscillator amb aquesta waveform)
        #         i self.env (un Envelope amb aquests quatre paràmetres)
        pass

    def nota(self, note_number, note_duration, velocity=100):
        # TODO 5: els 4 passos de l'enunciat
        return None  # <-- substitueix

In [ ]:
# AUTOTEST 3
veu = Veu(waveform='sawtooth')
# Si aquí surt AttributeError: 'Veu' object has no attribute 'osc' → revisa el TODO 4 (el self.!)
osc_inicial = veu.osc

n1 = veu.nota(60, 0.3, velocity=127)
assert n1 is not None, "nota() retorna None"
esperat = len(Envelope(0.02, 0.1, 0.6, 0.2).generate(0.3))
assert len(n1) == esperat, f"La nota hauria de tenir {esperat} mostres (les de la corba), en té {len(n1)}"
assert abs(veu.osc.freq - 261.63) < 0.01, "La freqüència de self.osc no correspon a la nota 60"
assert abs(n1[0]) < 0.05, "La nota hauria de començar en silenci (Attack)"
assert abs(n1[-1]) < 0.05, "La nota hauria d'acabar en silenci (Release)"
assert np.max(np.abs(n1)) > 0.7, "Amb velocity 127 l'amplitud màxima hauria de ser propera a 1"

n2 = veu.nota(64, 0.3, velocity=64)
assert veu.osc is osc_inicial, "La Veu ha de crear l'oscil·lador UN sol cop, a __init__ (no a cada nota)"
proporcio = np.max(np.abs(n2)) / np.max(np.abs(n1))
assert abs(proporcio - 64 / 127) < 0.05, "La velocity no s'aplica correctament"

print("✅ Part 4 correcta!")

In [ ]:
# 🔊 Prova-ho i escolta: la mateixa melodia amb dues veus
melodia = [60, 62, 64, 67, 69, 72]
suau = Veu('sine', attack=0.05, release=0.3)
aspra = Veu('sawtooth', attack=0.005, decay=0.05, sustain=0.4, release=0.1)

for veu in [suau, aspra]:
    trossos = []
    for nota in melodia:
        trossos.append(veu.nota(nota, 0.2, velocity=90))
    display(Audio(np.concatenate(trossos), rate=sample_rate))

# 👉 Crea una tercera veu amb el teu timbre, i prova una velocity diferent a cada nota.

## Part 5 — Munta-ho i escolta-ho

Crea una seqüència d'**almenys 6 notes** amb una o més `Veu`, que inclogui **almenys un cop** de `SamplePlayer`.

**Pistes:**
- Encadenar: `np.concatenate([tros1, tros2, ...])`, o ves afegint trossos a una llista dins un bucle (com a la cel·la 🔊 anterior)
- Que dues coses sonin **alhora**: cal que tinguin la mateixa longitud i sumar-les. Per exemple, un kick que duri el mateix que una nota: `kick.generate(len(nota) / sample_rate)`
- Si la suma passa de 1, normalitza: `so / np.max(np.abs(so))`

In [ ]:
# TODO 6: la teva seqüència

la_meva_sequencia = np.array([])   # <-- substitueix

In [ ]:
# AUTOTEST 4
assert len(la_meva_sequencia) > 1.0 * sample_rate, \
    "La seqüència sembla massa curta: assegura't que hi ha almenys 6 notes"
assert np.max(np.abs(la_meva_sequencia)) <= 1.0, \
    "La seqüència satura (passa de 1): normalitza-la"

print("✅ Part 5 correcta! Escolta-la a la cel·la següent.")

In [ ]:
# 🔊 La teva seqüència
Audio(la_meva_sequencia, rate=sample_rate)

---
## 🚀 Challenges (opcional)

1. **Acord (polifonia):** crea tres `Veu` i fes-les tocar tres notes diferents amb la mateixa durada; suma-les (i normalitza). Per què cal una `Veu` per nota i no en serveix una de sola?
2. **Envolvent percussiva:** una `Veu` amb `attack=0.001`, `decay=0.15`, `sustain=0` i release curt. Prova-la amb notes greus (MIDI 36-45). A què s'assembla?
3. **Un efecte com a classe:** escriu `RingMod(freq)`, que **conté** un `Oscillator` (la portadora) i té un mètode `process(so)` que retorna `so` × portadora (de la mateixa longitud que `so`). A la S05 calia la global `fase_ringmod`; aquí, qui guarda la fase? Prova'l aplicant-lo nota a nota a una melodia: la portadora continua sense salts? Per què el mètode es diu `process` i no `generate`?
4. **Un kit:** una classe `Kit` que guarda una **llista** de `SamplePlayer` i té un mètode `pas(index, duration)` que retorna el so del sample número `index`. Pots generar-te un segon sample (un charles de soroll curt amb envolvent, com a la Sessió 2) i desar-lo amb `sf.write`.

In [ ]:
# El teu codi dels challenges aquí